In [ ]:
%%writefile almacenamiento_profesional.yaml
apiVersion: v1
kind: PersistentVolume
metadata:
  name: pv-taller-compartido
spec:
  capacity:
    storage: 1Gi # Reservamos 1 Gigabyte de espacio
  volumeMode: Filesystem
  accessModes:
    - ReadWriteOnce # Significa que una máquina a la vez puede leer y escribir
  persistentVolumeReclaimPolicy: Retain # Si borramos el Job, los datos NO se destruyen
  hostPath:
    path: /run/desktop/mnt/host/wsl/datos_profesionales
    type: DirectoryOrCreate
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: claim-herramienta-python
spec:
  accessModes:
    - ReadWriteOnce
  resources:
    requests:
      storage: 1Gi # La manguera pide exactamente el Gigabyte que la pared ofrece

In [ ]:
%%bash
kubectl apply -f almacenamiento_profesional.yaml

In [ ]:
%%bash
kubectl get pv,pvc

In [ ]:
%%writefile almacenamiento_profesional.yaml
apiVersion: v1
kind: PersistentVolume
metadata:
  name: pv-taller-compartido
spec:
  capacity:
    storage: 1Gi
  volumeMode: Filesystem
  accessModes:
    - ReadWriteOnce
  persistentVolumeReclaimPolicy: Retain
  storageClassName: manual # <-- Le ponemos una etiqueta a la toma de pared
  hostPath:
    path: /run/desktop/mnt/host/wsl/datos_profesionales
    type: DirectoryOrCreate
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: claim-herramienta-python
spec:
  accessModes:
    - ReadWriteOnce
  storageClassName: manual # <-- Obligamos a la manguera a buscar solo la toma "manual"
  resources:
    requests:
      storage: 1Gi

In [ ]:
%%bash
kubectl delete pvc claim-herramienta-python --ignore-not-found=true
kubectl delete pv pv-taller-compartido --ignore-not-found=true

In [ ]:
%%bash
kubectl apply -f almacenamiento_profesional.yaml

In [ ]:
!kubectl get pv,pvc

In [ ]:
%%writefile job_profesional.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: job-almacenamiento-profesional
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      containers:
      - name: python-worker
        image: python:3.10-slim
        command: ["python", "/app/reporte.py"]
        env:
        - name: APP_ENV
          value: "Entorno-Arquitectura-Desacoplada"
        volumeMounts:
        # 1. Montamos el código del script (ConfigMap)
        - name: espacio-codigo
          mountPath: /app
        # 2. Conectamos la manguera virtual para que Python guarde los datos
        - name: carpeta-salida-profesional
          mountPath: /app/datos_reporte
      volumes:
      # Origen 1: El código limpio
      - name: espacio-codigo
        configMap:
          name: procesador-config
      # Origen 2: ¡La manguera de acople rápido (PVC)! No usamos hostPath aquí.
      - name: carpeta-salida-profesional
        persistentVolumeClaim:
          claimName: claim-herramienta-python # <-- Apuntamos directamente al Claim

In [ ]:
%%bash
kubectl apply -f job_profesional.yaml

In [ ]:
%%bash
kubectl get jobs

In [ ]:
%%bash
kubectl logs job/job-almacenamiento-profesional

In [ ]:
%%bash
kubectl delete job job-almacenamiento-profesional